# Week 17, Day 6: Forge: Week 17 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w017/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. bias = -sigma^2 / n =", -5 / 10)
print("2. I = n / lambda =", 20 / 4)
print("3. lambda = sigma^2 / tau^2 =", 2 ** 2 / 0.5 ** 2)

In [ ]:
rng = np.random.default_rng(2008)
T, dt, L, H = 20000, 0.005, 10, 6
s = rng.normal(size=T)
kt = np.arange(L)
k_true = 0.8 * np.exp(-kt / 3) * np.sin(kt / 2)
h_true = np.array([-4, -2, -0.8, 0.3, 0.3, 0.1])
b_true = np.log(20 * dt)
drive = b_true + np.convolve(s, k_true)[:T]            # stimulus part of the log-rate
y = np.zeros(T, int)
for t in range(T):
    past = y[max(0, t - H):t][::-1]                    # most recent spike count first
    y[t] = rng.poisson(np.exp(drive[t] + h_true[:len(past)] @ past))

def design(history):
    cols = [np.ones(T)] + [np.r_[np.zeros(j), s[:T - j]] for j in range(L)]
    if history:
        cols += [np.r_[np.zeros(j + 1), y[:T - j - 1]] for j in range(H)]
    return np.column_stack(cols)

train, test = np.arange(T) < 15000, np.arange(T) >= 15000
print(f"{y.sum()} spikes, mean rate {y.mean() / dt:.1f} Hz")

### Exercise: Part A: fit and compare

Write `fit_glm(X, y)` (Newton's method, 30 steps, starting from the log of the mean count in the intercept). Fit a stimulus-only model and a stimulus-plus-history model on the training data, then compute each one's test log-likelihood relative to a constant-rate model, in bits per spike. Store `w_stim`, `w_hist` and the dictionary `bits`.

In [ ]:
def fit_glm(X, yy, steps=30):
    w = np.zeros(X.shape[1]); w[0] = np.log(yy.mean())
    for _ in range(steps):
        lam = np.exp(X @ w)
        w = ...
    return w

def bits_per_spike(X, w):
    lam = np.exp(X[test] @ w)
    ll = np.sum(y[test] * np.log(lam) - lam)
    r0 = y[train].mean()                                   # constant-rate baseline fitted on training data
    ll0 = ...
    return (ll - ll0) / y[test].sum() / np.log(2)

w_stim = fit_glm(design(False)[train], y[train])
w_hist = fit_glm(design(True)[train], y[train])
bits = {"stimulus": bits_per_spike(design(False), w_stim), "history": bits_per_spike(design(True), w_hist)}
print({k: round(v, 3) for k, v in bits.items()})
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(k_true, "k", label="true"); ax[0].plot(w_hist[1:L + 1], "o-", label="fitted"); ax[0].set_title("stimulus filter"); ax[0].legend()
ax[1].plot(np.arange(1, H + 1), h_true, "k"); ax[1].plot(np.arange(1, H + 1), w_hist[L + 1:], "o-"); ax[1].set_title("spike-history filter"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "fit_glm converges: the gradient at the solution is tiny"
    X = design(True)[train]; lam = np.exp(X @ w_hist)
    assert np.abs(X.T @ (y[train] - lam)).max() < 1e-4

def test_history_filter():
    "the fitted spike-history filter recovers the refractory dip: first weight below -3"
    assert w_hist[L + 1] < -3

def test_bits():
    "the history model gains at least 0.2 bits per spike over the stimulus-only model on test data"
    assert bits["history"] - bits["stimulus"] > 0.2 and bits["stimulus"] > 0

test_fit()
test_history_filter()
test_bits()
print("All checks passed")

### Exercise: Part B: error bars from the curvature

Compute the negative Hessian of the log-likelihood at `w_hist` on the training data, invert it, and take the square roots of the diagonal as standard errors `se`. Count how many of the 17 true parameters fall within two standard errors.

In [ ]:
X = design(True)[train]
lam = np.exp(X @ w_hist)
neg_hess = ...
se = ...
truth = np.r_[b_true, k_true, h_true]
inside = np.abs(w_hist - truth) < 2 * se
print(f"{inside.sum()} of {len(truth)} true parameters within 2 standard errors")
plt.errorbar(np.arange(len(truth)), w_hist, yerr=2 * se, fmt="o", label="estimate ± 2 SE"); plt.plot(truth, "kx", label="true"); plt.legend(); plt.xlabel("parameter"); plt.show()

In [ ]:
import numpy as np
def test_se():
    "standard errors are positive and computed from the inverse negative Hessian"
    assert se.shape == w_hist.shape and np.all(se > 0)

def test_coverage():
    "at least 13 of the 17 true parameters lie within 2 standard errors of their estimates"
    truth = np.r_[b_true, k_true, h_true]
    assert np.sum(np.abs(w_hist - truth) < 2 * se) >= 13

test_se()
test_coverage()
print("All checks passed")

### Exercise: Part C: cross-validated regularisation

For λ in `lams` (13 values from 0.1 to 1,000), compute the 5-fold cross-validated prediction error of the smooth-prior estimate (`cv_curve`) and the true relative error of the estimate fitted on all 50 trials (`true_err`). Compare the cross-validated choice with the best.

In [ ]:
rng_c = np.random.default_rng(3)
P, n = 30, 50
pix = np.arange(P)
w_rf = np.exp(-(pix - 15) ** 2 / (2 * 4 ** 2)) * np.cos(2 * np.pi * (pix - 15) / 10)
Xs = rng_c.normal(size=(n, P)); ys = Xs @ w_rf + rng_c.normal(size=n)
D = np.diff(np.eye(P), axis=0)
def smooth_fit(Xa, ya, lam): return np.linalg.solve(Xa.T @ Xa + lam * D.T @ D, Xa.T @ ya)
lams = np.logspace(-1, 3, 13)
folds = np.array_split(rng_c.permutation(n), 5)

cv_curve, true_err = [], []
for lam in lams:
    errs = []
    for f in folds:
        tr = np.setdiff1d(np.arange(n), f)
        w = ...
        errs.append(...)
    cv_curve.append(np.mean(errs))
    w_all = smooth_fit(Xs, ys, lam)
    true_err.append(np.sum((w_all - w_rf) ** 2) / np.sum(w_rf ** 2))
cv_curve, true_err = np.array(cv_curve), np.array(true_err)
print(f"cross-validation picks lambda = {lams[np.argmin(cv_curve)]:.1f} (true error {true_err[np.argmin(cv_curve)]:.3f}); best lambda = {lams[np.argmin(true_err)]:.1f} (true error {true_err.min():.3f})")
plt.semilogx(lams, cv_curve / cv_curve.min(), label="cross-validated error (scaled)"); plt.semilogx(lams, true_err / true_err.min(), label="true error (scaled)"); plt.legend(); plt.xlabel("lambda"); plt.show()

In [ ]:
import numpy as np
def test_curve():
    "a cross-validated error is computed for each of the 13 values of lambda"
    assert len(cv_curve) == 13

def test_close():
    "the lambda chosen by cross-validation has a true error within 50% of the best lambda's"
    assert true_err[np.argmin(cv_curve)] < 1.5 * true_err.min()

test_curve()
test_close()
print("All checks passed")